# Serial-v0 interface 살펴보기

원본 `gym-invmgmt`의 `reset` / `step`이 무엇을 돌려주는지 보는 노트북입니다.

- **원본 interface**: `gym.make`, `reset`, `step`이 반환하는 `obs`, `reward`, `terminated`, `truncated`, `info`
- **우리가 추가한 출력**: 그 값을 STATUS / ACTION / REWARD로 풀어 쓰는 부분

환경은 과제에 맞게 `GymInvMgmt/Serial-v0` + `use_goodwill=True`입니다.

사용 순서:
1. 설정 셀을 실행합니다.
2. `reset` 셀을 실행합니다.
3. `ACTION` 숫자를 바꾸고 `step` 셀을 실행합니다.
4. 같은 에피소드를 이어가려면 `step`만 다시 실행합니다.
5. 처음부터 보려면 `reset`을 다시 실행합니다.

In [ ]:
from pathlib import Path
import sys

import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent  # notebooks/ 안에서 열면 한 칸 위로
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))  # walk_serial_steps.py 를 import 하려고

import gym_invmgmt  # noqa: F401  # GymInvMgmt/Serial-v0 등록
from inventory_rl.env import make_serial_env
from walk_serial_steps import print_environment, print_period

print("gym_invmgmt", gym_invmgmt.__version__)
print("프로젝트 경로", ROOT)

## 1. 환경 만들기

`inventory_rl.env.make_serial_env(demand="stationary")`를 사용합니다. 이 함수는 `DEMAND_PRESETS`에서 정상 수요 설정을 선택하여 원본 `gym.make`에 전달합니다.

In [ ]:
env = make_serial_env(demand="stationary")
core = env.unwrapped  # Wrapper 를 벗겨 X, Y, R 행렬에 바로 접근

print("환경 ID:", env.spec.id)
print("관측 공간:", env.observation_space)
print("행동 공간:", env.action_space)
print("행동 하한:", env.action_space.low)
print("행동 상한:", env.action_space.high)
print("행동 의미:")
print("  action[0]  유통 2 → 소매 1")
print("  action[1]  공장 3 → 유통 2")
print("  action[2]  원자재 4 → 공장 3")
print()
print_environment(core)

## 2. reset

원본은 `(obs, info)`를 돌려줍니다. 에피소드를 처음부터 다시 보려면 이 셀만 다시 실행하면 됩니다.

In [14]:
SEED = 42  # 수요 난수를 고정. 같은 시드면 같은 수요가 나옴
obs, info = env.reset(seed=SEED)  # obs 길이 15. 시작 재고 [100, 100, 200]

print("원본 reset 반환")
print("  obs 타입:", type(obs), "shape:", np.shape(obs))
print("  obs =", np.asarray(obs, dtype=float))
print("  info =", info)
print("  현재 기간 core.period =", core.period)

원본 reset 반환
  obs 타입: <class 'numpy.ndarray'> shape: (15,)
  obs = [  0.   0. 100. 100. 200.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   1.]
  info = {}
  현재 기간 core.period = 0


In [15]:
def decode_obs(obs):
    """원본 관측 벡터를 칸별로 풀어 본다. 학습에는 이 함수가 필요 없다."""
    obs = np.asarray(obs, dtype=float).reshape(-1)  # (15,)
    return {
        "직전 수요": obs[0],  # 예: 24.0
        "직전 미충족": obs[1],
        "재고 소매": obs[2],  # 시작 100
        "재고 유통": obs[3],  # 시작 100
        "재고 공장": obs[4],  # 시작 200
        "파이프라인 2→1 (0~3기간 후)": obs[5:9],  # L=4 일정 4칸
        "파이프라인 3→2 (0~3기간 후)": obs[9:13],
        "기간 t": obs[13],  # 0~29
        "sentiment": obs[14],  # 시작 1.0
    }

print("우리가 풀어 본 현재 STATUS")
for key, value in decode_obs(obs).items():
    print(f"  {key}: {value}")

우리가 풀어 본 현재 STATUS
  직전 수요: 0.0
  직전 미충족: 0.0
  재고 소매: 100.0
  재고 유통: 100.0
  재고 공장: 200.0
  파이프라인 2→1 (0~3기간 후): [0. 0. 0. 0.]
  파이프라인 3→2 (0~3기간 후): [0. 0. 0. 0.]
  기간 t: 0.0
  sentiment: 1.0


## 3. action 넣고 step

아래 숫자만 바꾼 뒤 `step` 셀을 실행합니다.

- `action[0]`: 유통 → 소매
- `action[1]`: 공장 → 유통
- `action[2]`: 원자재 → 공장

In [16]:
ACTION = [20.0, 20.0, 20.0]  # [유통→소매, 공장→유통, 원자재→공장]
action = np.asarray(ACTION, dtype=np.float64)
print("이번에 넣을 action =", action)

이번에 넣을 action = [20. 20. 20.]


In [17]:
period_before = core.period  # step 전에 읽어야 함. 끝나면 period 가 1 오른다
obs, reward, terminated, truncated, info = env.step(action)  # 다음 관측, 한 기간 이익

print("원본 step 반환")
print("  obs shape:", np.shape(obs))
print("  obs =", np.asarray(obs, dtype=float))
print("  reward =", float(reward))
print("  terminated =", terminated)
print("  truncated =", truncated)
print("  info =", info)
print()
print("우리가 풀어 본 다음 STATUS")
for key, value in decode_obs(obs).items():
    print(f"  {key}: {value}")
print()
print_period(core, period_before, action, float(reward), info)

원본 step 반환
  obs shape: (15,)
  obs = [ 24.   0.  76.  80. 200.   0.   0.   0.  20.   0.   0.   0.  20.   1.
   1.]
  reward = 31.62
  terminated = False
  truncated = False
  info = {'period': 0, 'step_reward': 31.62, 'total_inventory': np.float64(356.0), 'total_backlog': np.float64(0.0), 'sentiment': 1.0}

우리가 풀어 본 다음 STATUS
  직전 수요: 24.0
  직전 미충족: 0.0
  재고 소매: 76.0
  재고 유통: 80.0
  재고 공장: 200.0
  파이프라인 2→1 (0~3기간 후): [ 0.  0.  0. 20.]
  파이프라인 3→2 (0~3기간 후): [ 0.  0.  0. 20.]
  기간 t: 1.0
  sentiment: 1.0


기간 0

[STATUS]
------------------------------------------------------------
원본 관측이 담는 값
  직전 수요: 0.00
  직전 미충족: 0.00
  기간 t: 0
  sentiment: 1.000
보유 재고 X[t]
  노드 1 (소매): 100.00
  노드 2 (유통): 100.00
  노드 3 (공장): 200.00
이동 중 재고 Y[t]
  2->1 (L=4): 0.00
  3->2 (L=4): 0.00
  4->3 (L=0): 0.00
도착 일정 (관측 파이프라인, 0기간 후 = 이번 기간 입고)
  2->1: 합 0.00  [0기간 후 0.00, 1기간 후 0.00, 2기간 후 0.00, 3기간 후 0.00]
  3->2: 합 0.00  [0기간 후 0.00, 1기간 후 0.00, 2기간 후 0.00, 3기간 후 0.00]

[ACTION]
-------------------------

## 사용 방법

- 다른 발주량을 보려면 `ACTION = [10, 30, 20]`처럼 바꾸고 `ACTION` 셀과 `step` 셀을 다시 실행합니다.
- 30기간이 끝나면 `truncated=True`가 됩니다. 그때는 `reset`부터 다시 실행합니다.
- 학습 코드가 쓰는 값은 `obs`, `reward`, `terminated`, `truncated`입니다. 아래 STATUS/REWARD 산식은 이해를 위한 출력입니다.